In [0]:

import pandas as pd

# Caminhos dos arquivos originais
base = "/Volumes/workspace/default/stravl_dados/"

csv_path = base + "Stravl_Travel_Preference_Data.csv"
txt_path = base + "destination_ids.txt"

# Criar schemas, caso ainda não existam
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# BRONZE 1: Todos os viajantes e todas as colunas
df_viajantes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

(
    df_viajantes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.viajantes")
)

# BRONZE 2: Todos os destinos
# Preservar os índices originais de cada linha
with open(txt_path, "r", encoding="utf-8") as arquivo:
    linhas = arquivo.read().splitlines()

dados_destinos = [
    (indice, destino)
    for indice, destino in enumerate(linhas)
]

df_destinos = spark.createDataFrame(
    dados_destinos,
    ["destino_id", "destino"]
)

(
    df_destinos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.destinos")
)

# Conferência das tabelas completas
print("BRONZE CRIADA COM SUCESSO")
print("Viajantes:", spark.table("workspace.bronze.viajantes").count())
print("Destinos:", spark.table("workspace.bronze.destinos").count())
print("Colunas do CSV:", len(df_viajantes.columns))


In [0]:

from pyspark.sql import functions as F

# Ler as tabelas Bronze
viajantes = spark.table("workspace.bronze.viajantes")
destinos = spark.table("workspace.bronze.destinos")

# Verificar os campos disponíveis
print("Colunas dos viajantes:")
print(viajantes.columns)

# Conferir as listas de avaliações
display(
    viajantes.select(
        "id",
        "yes_swipes",
        "no_swipes",
        "maybe_swipes"
    ).limit(5)
)


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Caminhos dos arquivos
base = "/Volumes/workspace/default/stravl_dados/"

csv_path = base + "Stravl_Travel_Preference_Data.csv"
txt_path = base + "destination_ids.txt"

# 1. Importar o CSV de viajantes
df_viajantes = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

# 2. Salvar na Bronze
(
    df_viajantes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.viajantes")
)

print("Tabela de viajantes criada!")

# 3. Ler destinos preservando a ordem original
with open(txt_path, "r", encoding="utf-8") as arquivo:
    destinos = [
        (indice, linha.strip())
        for indice, linha in enumerate(arquivo)
        if linha.strip()
    ]

df_destinos = spark.createDataFrame(
    destinos,
    ["destino_id", "destino"]
)

# 4. Salvar destinos na Bronze
(
    df_destinos.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.destinos")
)

print("Tabela de destinos criada!")

# 5. Validar quantidades
print("Total de viajantes:", df_viajantes.count())
print("Total de destinos:", df_destinos.count())

display(df_viajantes.limit(5))
display(df_destinos.limit(10))



In [0]:

# Verificar o catálogo disponível
spark.sql("SELECT current_catalog()").show()

# Criar as camadas
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

print("Schemas criados com sucesso!")


# Caminho do Volume no Unity Catalog
caminho = "/Volumes/workspace/default/stravl_dados/"

# Listar arquivos disponíveis
arquivos = dbutils.fs.ls(caminho)

display(arquivos)
